<a href="https://colab.research.google.com/github/nejumi/wandb-physicalai-workshop/blob/main/notebooks/01_lerobot_wandb_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LeRobot × W&B — 20分 quickstart

ロボット実機や外部データを使わず、2次元到達タスクのデモを `LeRobotDataset` 形式で作り、小さな behavior cloning policy を学習します。

完了時には同じ W&B Run に次の3種類が揃います。

- Metrics: `train/loss`, `eval/final_distance`
- Model Artifact: `lerobot-tiny-bc-policy`
- Video: `eval/rollout`

Colab の **ランタイム > ランタイムのタイプを変更 > T4 GPU** を選び、上から順に実行してください。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import sys
print('Python:', sys.version)

## 1. 教材を準備する

初回だけ数分かかります。LeRobot 0.7 と W&B media 依存を入れます。

In [ ]:
from pathlib import Path
import os

repo = Path('/content/wandb-physicalai-workshop')
if not repo.exists():
    !git clone -q https://github.com/nejumi/wandb-physicalai-workshop.git {repo}
%cd {repo}
%pip install -q -e '.[quickstart]'

## 2. W&B にログインする

Colab の Secrets に `WANDB_API_KEY` を登録していれば自動で読みます。未登録なら入力欄が出ます。Dedicated Cloud の場合は `WANDB_BASE_URL` も Secrets に登録してください。

In [ ]:
import getpass, os
from google.colab import userdata
import wandb

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

api_key = secret('WANDB_API_KEY') or getpass.getpass('WANDB_API_KEY: ')
base_url = secret('WANDB_BASE_URL') or 'https://api.wandb.ai'
os.environ['WANDB_BASE_URL'] = base_url.rstrip('/')
os.environ['WANDB_PROJECT'] = secret('WANDB_PROJECT') or 'physical-ai-workshop'
entity = secret('WANDB_ENTITY')
if entity:
    os.environ['WANDB_ENTITY'] = entity
wandb.login(key=api_key, host=os.environ['WANDB_BASE_URL'], relogin=True)
print('project:', os.environ['WANDB_PROJECT'])

## 3. LeRobotDataset を作り、policy を学習する

青い点を赤い target に動かす expert trajectory を作り、その state → action を小さな MLP が模倣します。T4 では通常1分以内です。

In [ ]:
!python scripts/09_lerobot_colab_quickstart.py --steps 150

## 4. rollout を確認する

Notebook で動画を見た後、上の Run URL を開きます。Workspace の loss、Artifacts の model version、Media の rollout が同じ Run にあることを確認してください。

In [ ]:
from IPython.display import Video, display
display(Video('workspace/quickstart/lerobot/rollout.mp4', embed=True))